# Pediatric Chest Radiography Deep Diagnostic Monograph
## Deep Residual Learning & Explainable Grad-CAM for Pneumonia Screening

> **ElmasCore Medical AI Architecture** • Benchmark: **Kermany et al. (Cell 2018)** • Target Metric: **Sensitivity & ROC-AUC**

---

### Clinical Context & Radiographic Dilemma
Pneumonia remains a leading cause of pediatric morbidity and mortality worldwide. Accurate radiological diagnosis is critical:
1. **Healthy Parenchyma**: Clear lung fields without abnormal opacity or consolidation.
2. **Bacterial Pneumonia**: Typically presents with **focal lobar consolidation** (often localized in an upper or lower lobe), requiring prompt antibacterial therapy.
3. **Viral Pneumonia**: Manifests with **diffuse, bilateral interstitial opacities**, requiring supportive care rather than inappropriate antibacterial administration.

### Architectural Highlights (ElmasCore Standard)
- **Rigorous Stratification**: Resolves the historic 16-sample validation split artifact while preserving the 624-sample holdout test partition.
- **Class-Weighted Optimization**: Mitigates the 2.9:1 class imbalance to prevent under-detection of critical cases.
- **Youden's J Statistic**: Optimizes the clinical decision boundary ($J = \text{Sensitivity} + \text{Specificity} - 1$) to minimize fatal false negatives.
- **Explainable AI (Grad-CAM)**: Exposes activation heatmaps on the final convolutional layers to verify anatomical localization before clinical decision-making.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

# Register repository root into PYTHONPATH
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.core.config import PipelineConfig, get_default_config
from src.core.types import DiagnosisClass, EtiologyType, SplitName
from src.data_pipeline import build_dataset_manifest, build_eval_transforms, create_split_loaders
from src.interpretability import GradCAM, overlay_cam_on_image
from src.modeling import PneumoniaTrainer, build_classifier, build_loss_criterion
from src.validation import analyze_etiology_breakdown, evaluate_clinical_performance, format_clinical_audit_table

print("PyTorch version:", torch.__version__)
print("Device configured:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

### 1. Dataset Manifest & Cohort Stratification

In [ ]:
config = get_default_config()
manifest = build_dataset_manifest(
    data_dir=config.data_dir,
    use_resplit=True,
    val_ratio=0.10,
    random_seed=42,
)

print(f"Total Radiographs:     {len(manifest.records)}")
print(f"  - Train Partition:   {manifest.train_count}")
print(f"  - Validation:        {manifest.val_count}")
print(f"  - Test Holdout:      {manifest.test_count}")
print(f"Class Distribution:    Normal: {manifest.normal_count} | Bacteria: {manifest.bacteria_count} | Virus: {manifest.virus_count}")

### 2. Pathological Radiograph Inspection (Normal vs Bacterial vs Viral)

In [ ]:
test_records = [r for r in manifest.records if r.split == SplitName.TEST]
sample_normal = next(r for r in test_records if r.etiology == EtiologyType.NORMAL)
sample_bacteria = next(r for r in test_records if r.etiology == EtiologyType.BACTERIA)
sample_virus = next(r for r in test_records if r.etiology == EtiologyType.VIRUS)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, rec, title in zip(
    axes,
    [sample_normal, sample_bacteria, sample_virus],
    ["Normal (Clear Parenchyma)", "Bacterial (Focal Consolidation)", "Viral (Diffuse Interstitial)"]
):
    with Image.open(rec.path) as img:
        ax.imshow(img.convert("RGB"))
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.axis("off")
plt.tight_layout()
plt.show()

### 3. Model Architecture & Clinical Inference Evaluation

In [ ]:
model = build_classifier(config)
checkpoint_path = config.checkpoints_dir / "best_model.pth"

if checkpoint_path.exists():
    print(f"Loading trained weights from {checkpoint_path}")
    model.load_state_dict(torch.load(checkpoint_path, map_location=config.device))
else:
    print("Model weights not yet trained; run main.py or trainer.fit().")

train_loader, val_loader, test_loader = create_split_loaders(manifest, config)

# Evaluate on 624-patient test holdout
model.eval()
all_targets, all_probs = [], []
with torch.no_grad():
    for images, labels, _ in test_loader:
        images = images.to(config.device)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)[:, 1]
        all_targets.extend(labels.cpu().tolist())
        all_probs.extend(probs.cpu().tolist())

eval_result = evaluate_clinical_performance(np.array(all_targets), np.array(all_probs), threshold=0.5)
etiology_stats = analyze_etiology_breakdown(test_records, all_probs, threshold=0.5)
print(format_clinical_audit_table(eval_result, etiology_stats))

### 4. Explainable AI: Grad-CAM Activation Heatmaps
Visualizing whether the deep network focuses on actual thoracic pathology rather than background artifacts.

In [ ]:
target_layer = model.get_target_layer_for_cam()
grad_cam = GradCAM(model, target_layer)
eval_tf = build_eval_transforms(config.image_size, config.mean, config.std)

fig, axes = plt.subplots(3, 2, figsize=(10, 12))
samples = [sample_normal, sample_bacteria, sample_virus]

for i, rec in enumerate(samples):
    with Image.open(rec.path) as img:
        rgb_img = img.convert("RGB")
        tensor = eval_tf(rgb_img).unsqueeze(0).to(config.device)
        heatmap = grad_cam.generate_cam(tensor, target_class=1)
        overlay = overlay_cam_on_image(rgb_img, heatmap, alpha=0.45)

    axes[i, 0].imshow(rgb_img)
    axes[i, 0].set_title(f"Original: {rec.etiology.value}", fontsize=11, fontweight="bold")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(overlay)
    axes[i, 1].set_title(f"Grad-CAM Pathological Saliency", fontsize=11, fontweight="bold")
    axes[i, 1].axis("off")

grad_cam.release()
plt.tight_layout()
plt.show()